# 75 - Latent Q-guidance with three Q50 critics on held-out PRO160

This single full worker tests whether each frozen Q50 critic can guide one ordinary PI0.5 sample by differentiating Q through the current clean-action estimate and taking one normalized ascent step on the live flow latent. The three arms use the original Q50, failure-prioritized Q50, and 4-chunk-U20-prioritized Q50.

All arms start from the exact stock noise, use the ordinary 10-step flow decoder, update once at zero-based Euler step 3 with latent RMS 0.005, execute the first 10 of 50 actions, and replan. This is deliberately not the paper's 64-candidate Q-softmax planner.

For diagnosis, every boundary also decodes the exact same-noise stock chunk and logs pre/post Q plus first-10/full-50 action displacement. It never substitutes a historical action into the live trajectory. Periodic success tables reuse only the exact stored stock outcome for each matched identity from notebook 68. Videos, frames, and generated chunks are off; reruns resume exact completed rows.

## 1. Setup

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Resolve the three checkpoints

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

EPISODE_LIMIT = None  # set to 1 for a three-rollout smoke test
LATENT_UPDATE_RMS = 0.005
ORIGINAL_ROOT = Path('/content/drive/MyDrive/pnp_qplanning_corrector')
PRIORITY_ROOT = Path('/content/drive/MyDrive/pnp_qplanning_priority')
ORIGINAL_Q50_CHECKPOINT_PATH = None
FAILURE_CHECKPOINT_PATH = None
U20_4CHUNK_CHECKPOINT_PATH = None

def resolve_one(root, pattern, explicit, label):
    if explicit is not None:
        path = Path(explicit)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(root.glob(pattern))
    if len(matches) != 1:
        raise ValueError(
            f'Expected exactly one {label} checkpoint; found {len(matches)}: ' +
            f'{[str(path) for path in matches]}. Set its path explicitly.')
    return matches[0]

ORIGINAL_Q50_CHECKPOINT_PATH = resolve_one(
    ORIGINAL_ROOT, 'pcpcds-*/q50_full/checkpoint_step_008000.pt',
    ORIGINAL_Q50_CHECKPOINT_PATH, 'original Q50 step-8000')
FAILURE_CHECKPOINT_PATH = resolve_one(
    PRIORITY_ROOT, 'pcpcds-*/q50_priority_failure_full/checkpoint_step_006000.pt',
    FAILURE_CHECKPOINT_PATH, 'failure-priority Q50 step-6000')
U20_4CHUNK_CHECKPOINT_PATH = resolve_one(
    PRIORITY_ROOT, 'pcpcds-*/q50_priority_u20_4chunk_full/checkpoint_step_006000.pt',
    U20_4CHUNK_CHECKPOINT_PATH, '4-chunk-U20-priority Q50 step-6000')

print({
    'worker': 'single full PRO160 worker',
    'identities': 160 if EPISODE_LIMIT is None else EPISODE_LIMIT,
    'new_rollouts': 480 if EPISODE_LIMIT is None else 3 * EPISODE_LIMIT,
    'original_q50': str(ORIGINAL_Q50_CHECKPOINT_PATH),
    'failure_priority_q50': str(FAILURE_CHECKPOINT_PATH),
    'u20_4chunk_priority_q50': str(U20_4CHUNK_CHECKPOINT_PATH),
    'latent_update_rms': LATENT_UPDATE_RMS,
    'periodic_print_every_complete_identities': 10,
    'historical_stock': 'reuse exact matched notebook-68 outcomes; do not rerun',
})

## 3. Run all three latent-guidance arms

In [ ]:
from pnp.qplanning_gradient_eval_experiment import (
    run_qplanning_latent_guidance_heldout160)

report = run_qplanning_latent_guidance_heldout160(
    original_checkpoint_path=ORIGINAL_Q50_CHECKPOINT_PATH,
    failure_checkpoint_path=FAILURE_CHECKPOINT_PATH,
    u20_4chunk_checkpoint_path=U20_4CHUNK_CHECKPOINT_PATH,
    episode_limit=EPISODE_LIMIT,
    update_rms=LATENT_UPDATE_RMS,
)
report

## 4. Audit persisted guidance settings and telemetry

In [ ]:
from pnp.qplanning_gradient_eval_experiment import (
    validate_qplanning_latent_guidance_sentinel)

validate_qplanning_latent_guidance_sentinel(
    checkpoint_ids=report['checkpoint_ids'],
    experiment=report['experiment'],
)